In [1]:
from __future__ import print_function
import sys, os, glob, subprocess
import math
import multiprocessing

import ROOT
from ROOT import gROOT, gStyle, gDirectory, gPad
nthreads = min(4, max(1, multiprocessing.cpu_count()-1))  # tune as needed
ROOT.EnableImplicitMT(nthreads)
ROOT.gStyle.SetPalette(103)
ROOT.gStyle.SetNumberContours(255)

from pdgid import pdgid_dict
import plot_utils
import constants
import log_parser

pdgId_list = [-11, 11, 22, 2112, 2212, 1000010020, 1000010030]
id_names   = ["positron","electron","photon","neutron","proton","D","T"]

tmp_dir = "/exp/mu2e/data/users/yongyiwu/stm_analysis/tmp"
run_types = ["MuBeamStopVD", "MuminusStopToVD"]
hist_files = []
for run_type in run_types:
    hist_path = os.path.join(tmp_dir, f"hist_{run_type}_merged.root")
    hist_files.append(hist_path)

nStoppedMuminus_MuBeamStopVD = 107501479. # from running log_parser.py
nStoppedMuminus_MuminusStopToVD = 991.*100000.
# weight normalize to 1e8 stopped muons
weight_MuBeamStopVD = 1000.*100000./nStoppedMuminus_MuBeamStopVD
weight_MuminusStopToVD = 1000.*100000./nStoppedMuminus_MuminusStopToVD
weights = [weight_MuBeamStopVD, weight_MuminusStopToVD]

pdf_name = "nucap_hist.pdf"
c = ROOT.TCanvas("c", "c", 1600, 1000)
c.Print(pdf_name + "[")

for pdgId, pid in zip(pdgId_list, id_names):
    f = [ROOT.TFile.Open(hist_file, "READ") for hist_file in hist_files]
    hist_types = ["pz", "pt", "x", "y", "startz", "E"]
    if pid == "photon":
        hist_types.extend(["E_zoom1","E_zoom2","E_zoom3"])
    elif pid == "neutron":
        hist_types.extend(["pz_zoom","pt_zoom","E_zoom"])
    hist_types.extend(["pz_vs_r", "pt_vs_r", "pt_vs_pz", "y_vs_x"])
    hist_dict = {}
    for hist_name in hist_types:
        hist_dict.update({hist_name : [f[i].Get(f"{run_type}_{pid}_{hist_name}") for (i, run_type) in enumerate(run_types)]})
        
        c0 = ROOT.TCanvas(f"c0_{pid}_{hist_name}", f"c_{hist_name}", 1600, 1000)
        gStyle.SetOptStat(1111110)
        c0.SetLeftMargin(0.12 if hist_dict[hist_name][0].InheritsFrom("TH2") else 0.1)
        c0.SetRightMargin(0.35 if hist_dict[hist_name][0].InheritsFrom("TH2") else 0.3)
        if pdgId in [22, 2112] and hist_name in ["pz", "pt", "E", "E_zoom1","E_zoom2","E_zoom3", "pz_zoom", "pt_zoom", "E_zoom"]:
            c0.SetLogy()
        else:
            c0.SetLogy(0)
        
        hist_dict[hist_name][0].Draw("colz" if hist_dict[hist_name][0].InheritsFrom("TH2") else "HIST")
        c0.Update()
        c0.Print(pdf_name, f"Title: {run_types[0]}_{pid}_{hist_name}")
        
        c1 = ROOT.TCanvas(f"c1_{pid}_{hist_name}", f"c_{hist_name}", 1600, 1000)
        gStyle.SetOptStat(1111110)
        c1.SetLeftMargin(0.12 if hist_dict[hist_name][0].InheritsFrom("TH2") else 0.1)
        c1.SetRightMargin(0.35 if hist_dict[hist_name][0].InheritsFrom("TH2") else 0.3)
        if pdgId in [22, 2112] and hist_name in ["pz", "pt", "E", "E_zoom1","E_zoom2","E_zoom3", "pz_zoom", "pt_zoom", "E_zoom"]:
            c1.SetLogy()
        else:
            c1.SetLogy(0)
            
        hist_dict[hist_name][1].Draw("colz" if hist_dict[hist_name][0].InheritsFrom("TH2") else "HIST")
        c1.Update()
        c1.Print(pdf_name, f"Title: {run_types[1]}_{pid}_{hist_name}")

        if not hist_dict[hist_name][0].InheritsFrom("TH2"):
            hist_ratio = hist_dict[hist_name][0].Clone()
                        
            c2 = ROOT.TCanvas(f"c2_{pid}_{hist_name}", f"c_{hist_name}", 1600, 1000)
            c2.SetLeftMargin(0.1)
            c2.SetRightMargin(0.3)
            if pdgId in [22, 2112] and hist_name in ["pz", "pt", "E", "E_zoom1","E_zoom2","E_zoom3", "pz_zoom", "pt_zoom", "E_zoom"]:
                c2.SetLogy()
            else:
                c2.SetLogy(0)

            title = hist_dict[hist_name][0].GetTitle().split(";")[0]
            newtitle = title.split(":")[1]
            hist_dict[hist_name][0].SetTitle(newtitle)
            hist_dict[hist_name][0].SetMaximum(1.1*max(hist_dict[hist_name][0].GetMaximum(), hist_dict[hist_name][1].GetMaximum()))
            
            hist_dict[hist_name][0].SetStats(0)
            hist_dict[hist_name][0].SetLineWidth(1)
            hist_dict[hist_name][0].SetLineColor(2)
            hist_dict[hist_name][0].SetMarkerColor(2)
            hist_dict[hist_name][0].Draw("")
            hist_dict[hist_name][1].SetStats(0)
            hist_dict[hist_name][1].SetLineWidth(1)
            hist_dict[hist_name][1].SetLineColor(1)
            hist_dict[hist_name][1].SetMarkerColor(1)
            hist_dict[hist_name][1].Draw("SAME")
            legend2 = ROOT.TLegend(0.75,0.7,0.95,0.9);
            legend2.AddEntry(hist_dict[hist_name][0],"#splitline{Geant4 stock}{"+f"{weights[0]*hist_dict[hist_name][0].GetEntries():.0f}"+" entries}","le")
            legend2.AddEntry(hist_dict[hist_name][1],"#splitline{Mu2e resampler}{"+f"{weights[1]*hist_dict[hist_name][1].GetEntries():.0f}"+" entries}","le")
            legend2.Draw("SAME")
            
            c2.Update()
            c2.Print(pdf_name, f"Title: comparison_{pid}_{hist_name}_0")

            c3 = ROOT.TCanvas(f"c3_{pid}_{hist_name}", f"c_{hist_name}", 1600, 1000)
            c3.SetLeftMargin(0.1)
            c3.SetRightMargin(0.3)
            hist_dict[hist_name][0].Sumw2(); hist_dict[hist_name][1].Sumw2()
            hist_ratio.Divide(hist_dict[hist_name][1])
            if hist_ratio.GetMaximum()>10.:
                c3.SetLogy()

            newtitle = "ratio of G4 stock / muon resampler: "+newtitle
            hist_ratio.SetTitle(newtitle)
            hist_ratio.GetYaxis().SetTitle("ratio") 
            hist_ratio.SetStats(0)
            hist_ratio.SetLineWidth(2)
            hist_ratio.SetLineColor(4)
            hist_ratio.SetMarkerColor(4)
            hist_ratio.Draw("")

            xmin = hist_ratio.GetXaxis().GetXmin()
            xmax = hist_ratio.GetXaxis().GetXmax()
            line = ROOT.TLine(xmin, 1.0, xmax, 1.0)
            line.SetLineColor(12)
            line.SetLineWidth(1)
            line.SetLineStyle(2)   # dashed
            line.Draw("SAME")
            legend3 = ROOT.TLegend(0.75,0.7,0.95,0.9);
            legend3.AddEntry(hist_ratio,"#splitline{G4 stock/mu resampler}{= "+f"{weights[0]*hist_dict[hist_name][0].GetEntries()/weights[1]/hist_dict[hist_name][1].GetEntries():.3f}"+"}","le")
            legend3.Draw("SAME")

            c3.Update()
            c3.Print(pdf_name, f"Title: comparison_{pid}_{hist_name}_1")
            
    for tf in f:
        tf.Close()

c.Print(pdf_name + "]")

Info in <TCanvas::Print>: pdf file nucap_hist.pdf has been created
Info in <TCanvas::Print>: Current canvas added to pdf file nucap_hist.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file nucap_hist.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file nucap_hist.pdf
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Info in <TCanvas::Print>: Current canvas added to pdf file nucap_hist.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file nucap_hist.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file nucap_hist.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file nucap_hist.pdf
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Warning in <TH1D::Sumw2>: Sum of squares of weights structure already created
Info in <TCanvas::Print>: Current canvas added to pdf file nucap_hist.pdf
Info in <TCanvas::Print>: Cur